### Load Data and Preprocess

In [21]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# 1. Load and Preprocess the Dataset (based on EX7)
df = pd.read_csv('credit.csv')
df.head()

,checking_balance,months_loan_duration,credit_history,purpose,amount,savings_balance,employment_duration,percent_of_income,years_at_residence,age,other_credit,housing,existing_loans_count,job,dependents,phone,default
0,< 0 DM,6,critical,furniture/appliances,1169,unknown,> 7 years,4,4,67,none,own,2,skilled,1,yes,no
1,1 - 200 DM,48,good,furniture/appliances,5951,< 100 DM,1 - 4 years,2,2,22,none,own,1,skilled,1,no,yes
2,unknown,12,critical,education,2096,< 100 DM,4 - 7 years,2,3,49,none,own,1,unskilled,2,no,no
3,< 0 DM,42,good,furniture/appliances,7882,< 100 DM,4 - 7 years,2,4,45,none,other,1,skilled,2,no,no
4,< 0 DM,24,poor,car,4870,< 100 DM,1 - 4 years,3,4,53,none,other,2,skilled,2,no,yes


In [22]:
# Label Encoding for the target variable 'default'
df['default'] = LabelEncoder().fit_transform(df['default'])

# One-Hot Encoding for categorical features
df_encoded = pd.get_dummies(df, drop_first=True)

# Randomize the dataset
df_encoded = df_encoded.sample(frac=1, random_state=42).reset_index(drop=True)

In [23]:
# Split the data into training and testing sets (80-20 split)
X = df_encoded.drop('default', axis=1)
y = df_encoded['default']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Define Parameter Grids and Apply GridSearchCV

In [24]:
# 2. Define Models and Parameter Grids for GridSearchCV
models = {
    'Random Forest': (
        RandomForestClassifier(random_state=42),
        {'n_estimators': [50, 100, 200], 'max_depth': [3, 5, 7, 10]}
    ),
    'Bagging Classifier': (
        BaggingClassifier(random_state=42),
        {'n_estimators': [10, 50, 100], 'max_samples': [0.5, 0.8, 1.0]}
    ),
    'AdaBoost Classifier': (
        AdaBoostClassifier(random_state=42),
        {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 1.0]}
    )
}

best_estimators = {}
results = []

In [25]:
print("=== Training Base Models via GridSearchCV ===")
# Perform GridSearchCV for each base ensemble model
for name, (model, param_grid) in models.items():
    grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    grid_search.fit(X_train, y_train)
    # Get the best estimator and extract predictions
    best_model = grid_search.best_estimator_
    best_estimators[name] = best_model
    y_pred = best_model.predict(X_test)

    # Calculate accuracy and ROC-AUC
    acc = accuracy_score(y_test, y_pred)
    if hasattr(best_model, "predict_proba"):
        y_prob = best_model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_prob)
    else:
        roc_auc = "N/A"

    results.append({
        'Model': name,
        'Best Params': grid_search.best_params_,
        'Test Accuracy': round(acc, 4),
        'ROC-AUC': round(roc_auc, 4) if isinstance(roc_auc, float) else roc_auc
    })

=== Training Base Models via GridSearchCV ===


### Implement Hard and Soft Voting

In [26]:
# 3. Implement Voting Classifiers (Using the best base estimators)
estimators = [(name, model) for name, model in best_estimators.items()]

# Hard Voting
voting_hard = VotingClassifier(estimators=estimators, voting='hard')
voting_hard.fit(X_train, y_train)
acc_hard = accuracy_score(y_test, voting_hard.predict(X_test))

results.append({
    'Model': 'Voting Classifier (Hard)',
    'Best Params': 'N/A',
    'Test Accuracy': round(acc_hard, 4),
    'ROC-AUC': 'N/A' # Hard Voting only predicts class labels, not probabilities
})

In [28]:
# Soft Voting
voting_soft = VotingClassifier(estimators=estimators, voting='soft')
voting_soft.fit(X_train, y_train)
acc_soft = accuracy_score(y_test, voting_soft.predict(X_test))
y_prob_soft = voting_soft.predict_proba(X_test)[:, 1]
roc_auc_soft = roc_auc_score(y_test, y_prob_soft)

results.append({
    'Model': 'Voting Classifier (Soft)',
    'Best Params': 'N/A',
    'Test Accuracy': acc_soft,
    'ROC-AUC': round(roc_auc_soft, 4)
})

### Display the Results

In [29]:
# 4. Summary of Results
results_df = pd.DataFrame(results)
print("\n=== Model Performance Comparison ===")
print(results_df.to_string(index=False))


=== Model Performance Comparison ===
                   Model                                Best Params  Test Accuracy ROC-AUC
           Random Forest      {'max_depth': 10, 'n_estimators': 50}          0.695  0.7213
      Bagging Classifier  {'max_samples': 0.8, 'n_estimators': 100}          0.720  0.7365
     AdaBoost Classifier {'learning_rate': 1.0, 'n_estimators': 50}          0.685  0.7142
Voting Classifier (Hard)                                        N/A          0.710     N/A
Voting Classifier (Soft)                                        N/A          0.695  0.7331


### Summary of Findings
#### Model Training and Evaluation Discoveries
During the evaluation of multiple ensemble methods (Random Forest, Bagging, AdaBoost, and Voting Classifiers) on the credit default dataset, the Bagging Classifier emerged as the most effective approach. It achieved the highest test accuracy (72.00%) and the strongest ROC-AUC score (0.7365). Conversely, AdaBoost struggled the most, yielding only a 68.00% accuracy.<br><br>

#### Performance Changes After Tuning
Transitioning from the single Decision Tree baseline in the EX7 notebook to ensemble methods significantly improved model robustness by reducing variance. Systematic hyperparameter tuning via GridSearchCV helped pinpoint optimal configurations for each model (e.g., Bagging performed best with 100 estimators and an 80% sample rate). Despite utilizing these advanced techniques, overall accuracy hit a "predictive ceiling" around 72–73%. Furthermore, combining the tuned models into Hard and Soft Voting Classifiers (71.00%) did not outpace the standalone Bagging model, likely because the ensemble was dragged down by AdaBoost's weaker predictions.<br><br>

#### Key Insights, Challenges, and Lessons Learned
- **The Recall Challenge**: In credit risk, minimizing False Negatives (approving a bad loan) is critical. Despite solid accuracy and ROC-AUC scores, all models struggled with Recall (peaking around ~40%). This highlights a persistent challenge in identifying all actual defaulters using the current feature set.

- **Interpretability vs. Performance**: While ensembles like Bagging offer better probability estimates and stability than a single Decision Tree, they operate as "black boxes." In the heavily regulated banking industry, sacrificing the plain-language explainability of a single tree for marginal performance gains presents a significant real-world implementation hurdle.

- **Feature Limitations**: The persistent performance plateau across both simple trees and complex ensembles suggests that the provided demographic and account balance features have a natural predictive limit. Accurately modeling human financial behavior likely requires richer, more complex behavioral data.